<a href="https://colab.research.google.com/github/dquispec/IA/blob/main/RAG_Noticias_Peru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema RAG sobre Noticias de Perú (Andina)
**Proyecto Final — IA Generativa G2**

Pipeline: Andina (RSS+scraping) → Chunking → Embeddings → FAISS → Retrieval → Qwen2.5-3B-Instruct → Grounding → Evaluación

Idioma del corpus: **español**. Ejecutar en **Google Colab con GPU T4**.

## 1. Instalación de librerías
**IMPORTANTE:** esta celda reinicia el kernel al final. Espera el reinicio y continúa desde la celda 1.1.

In [ ]:
!pip -q install -U sentence-transformers faiss-cpu transformers accelerate bitsandbytes feedparser beautifulsoup4 lxml requests nltk
import os
os.kill(os.getpid(), 9)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 w

### 1.1 Tokenizadores NLTK (ejecutar tras el reinicio)

In [1]:
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)

True

## 2. Descarga del dataset — Agencia Andina (Perú)

**Fuente:** Andina, agencia oficial de noticias del Perú (andina.pe).
**Dominio:** noticias en español — política, economía, sociedad nacional.
**Método:** RSS feeds públicos + parser HTML para extraer cuerpo completo.

In [2]:
import feedparser, requests, re, time
from bs4 import BeautifulSoup

FEEDS = [
    'https://andina.pe/agencia/rss.aspx?id=1',
    'https://andina.pe/agencia/rss.aspx?id=2',
    'https://andina.pe/agencia/rss.aspx?id=3',
]
UA = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'}

def fetch_article(url):
    try:
        r = requests.get(url, headers=UA, timeout=20)
        if r.status_code != 200:
            return None
        soup = BeautifulSoup(r.text, 'lxml')
        body = soup.find('div', {'class': re.compile(r'(Descripcion|contenido|article-body|nota-cuerpo|cuerpo|content-nota)', re.I)})
        if not body:
            art = soup.find('article')
            if art:
                paras = art.find_all('p')
                text = ' '.join(p.get_text(' ', strip=True) for p in paras)
                return text if len(text) > 200 else None
        if not body:
            return None
        for s in body(['script','style','figure','aside','iframe']):
            s.decompose()
        text = re.sub(r'\s+', ' ', body.get_text(' ', strip=True)).strip()
        return text if len(text) > 200 else None
    except:
        return None

papers = []
seen = set()
for feed_url in FEEDS:
    f = feedparser.parse(feed_url)
    print(f"Feed {feed_url[-1]}: {len(f.entries)} entradas")
    for e in f.entries:
        if len(papers) >= 120: break
        url = e.link
        if url in seen: continue
        seen.add(url)
        rss_desc = re.sub(r'<[^>]+>', '', getattr(e, 'summary', '') or '').strip()
        body = fetch_article(url)
        text = body if body else (rss_desc if len(rss_desc) > 100 else None)
        if not text: continue
        papers.append({
            'id': url.split('/')[-1][:60],
            'title': e.title.strip(),
            'abstract': text[:3000],
            'published': e.get('published',''),
            'url': url
        })
        time.sleep(0.3)
    if len(papers) >= 120: break

papers = papers[:110]
print(f'Descargadas {len(papers)} noticias')

Feed 1: 0 entradas
Feed 2: 20 entradas
Feed 3: 20 entradas
Descargadas 40 noticias


### 2.1 Exploración del dataset

In [3]:
import pandas as pd
df = pd.DataFrame(papers)
df['len'] = df['abstract'].str.len()
print(df[['title','len','published']].head(5).to_string(index=False))
print('\nTotal:', len(df))
print('Longitud media:', int(df['len'].mean()))

                                                                                  title  len                       published
              Educación y empleo: claves para integrar a personas refugiadas en el Perú  186 Sat, 23 May 2026 15:00:01 -0500
 Banco Mundial: Más del 80% de venezolanos en Perú se encuentran económicamente activos  197 Sat, 23 May 2026 13:49:29 -0500
         Desconfianza e informalidad aún frenan el crecimiento del comercio electrónico  448 Sat, 23 May 2026 11:00:00 -0500
   ProInversión y gremios privados impulsan infraestructura de alto impacto en Arequipa  499 Sat, 23 May 2026 10:00:00 -0500
SBS deja sin efecto régimen de intervención de la Cooperativa de Ahorro y Crédito Kuria  279 Sat, 23 May 2026 09:00:00 -0500

Total: 40
Longitud media: 311


In [4]:
print(len(papers))
print(papers[0] if papers else "VACÍO")

40
{'id': 'noticia-educacion-y-empleo-claves-para-integrar-a-personas-r', 'title': 'Educación y empleo: claves para integrar a personas refugiadas en el Perú', 'abstract': 'El acceso a documentación, servicios, educación y empleo formal permite que las personas refugiadas reconstruyan sus vidas y contribuyan al desarrollo de las comunidades que las reciben.', 'published': 'Sat, 23 May 2026 15:00:01 -0500', 'url': 'https://andina.pe/agencia/noticia-educacion-y-empleo-claves-para-integrar-a-personas-refugiadas-el-peru-1076345.aspx'}


### 2.2 Fuente, dominio y dificultades

**Fuente:** Andina (andina.pe), agencia oficial del Estado peruano. RSS público + scraping HTML.

**Dominio:** noticias en español del Perú — política, economía, sociedad, regional, internacional.

**Dificultades encontradas:**
- HTML inconsistente entre artículos (varias plantillas).
- Anti-bot suave: requiere `User-Agent` válido.
- Rate limit informal: `time.sleep(0.5)` entre requests.
- Encoding UTF-8 con acentos y ñ — manejado por `requests`.
- Contenido perecedero: el dataset queda fijado a la fecha de scraping.
- Algunos artículos con cuerpo vacío (videos, galerías) — filtrados por longitud mínima 200.

## 3. Chunking + Overlap — comparación 128 vs 256

In [5]:
def chunk_text(text, size, overlap):
    words = text.split()
    chunks = []
    step = size - overlap
    for i in range(0, len(words), step):
        c = ' '.join(words[i:i+size])
        if len(c.split()) >= 30:
            chunks.append(c)
        if i+size >= len(words):
            break
    return chunks

def build_corpus(papers, size, overlap):
    out = []
    for p in papers:
        full = f"{p['title']}. {p['abstract']}"
        for j, ch in enumerate(chunk_text(full, size, overlap)):
            out.append({
                'paper_id': p['id'],
                'title': p['title'],
                'chunk_id': f"{p['id']}#{j}",
                'text': ch,
                'url': p['url']
            })
    return out

corpus_small = build_corpus(papers, size=128, overlap=24)
corpus_large = build_corpus(papers, size=256, overlap=48)
print(f'Chunks 128/24: {len(corpus_small)}')
print(f'Chunks 256/48: {len(corpus_large)}')

Chunks 128/24: 40
Chunks 256/48: 40


### 3.1 Análisis de chunking

**Tradeoff tamaño vs fragmentación:**
- Chunks 128 → granularidad alta, mejor para datos puntuales (nombres, cifras, fechas).
- Chunks 256 → más contexto, mejor para resumen o causa-efecto.

**Impacto en recuperación:**
- Noticias suelen tener pirámide invertida (lo importante al inicio). Chunks 256 capturan el "qué/quién/dónde/cuándo" completo.
- Chunks 128 fragmentan demasiado el lead, perdiendo contexto.

**Rol del overlap:**
- En noticias, cifras y nombres aparecen una sola vez. Overlap evita perder esa pieza si cae en borde de chunk.

## 4. Embeddings — multilingual-e5-base
Modelo multilingüe, ideal para español sin necesidad de uno específico.

In [6]:
from sentence_transformers import SentenceTransformer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
emb_model = SentenceTransformer('intfloat/multilingual-e5-base', device=device)

def embed_passages(texts, batch=64):
    return emb_model.encode([f'passage: {t}' for t in texts], batch_size=batch,
                            show_progress_bar=True, normalize_embeddings=True)

def embed_query(q):
    return emb_model.encode([f'query: {q}'], normalize_embeddings=True)[0]

emb_small = embed_passages([c['text'] for c in corpus_small])
emb_large = embed_passages([c['text'] for c in corpus_large])
print('Shape large:', emb_large.shape)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Shape large: (40, 768)


### 4.1 Justificación

**Modelo:** `intfloat/multilingual-e5-base`. 768-dim, multilingüe (100+ idiomas), entrenado con contrastive learning.

**Por qué para español:** rinde mejor que modelos solo-EN traducidos. Prefijos `query:`/`passage:` mejoran retrieval asimétrico.

**Qué representan:** vectores densos donde textos con significado similar quedan cerca en espacio coseno.

**Por qué en RAG:** permiten buscar por significado, no por palabra exacta. Útil cuando la pregunta usa sinónimos o paráfrasis del texto fuente.

## 5. Vector Store — FAISS Flat + HNSW (BONUS)

In [7]:
import faiss

dim = emb_large.shape[1]

index_flat = faiss.IndexFlatIP(dim)
index_flat.add(emb_large.astype('float32'))

index_hnsw = faiss.IndexHNSWFlat(dim, 32)
index_hnsw.hnsw.efConstruction = 80
index_hnsw.hnsw.efSearch = 64
index_hnsw.add(emb_large.astype('float32'))

index_small = faiss.IndexFlatIP(dim)
index_small.add(emb_small.astype('float32'))

print(f'Flat: {index_flat.ntotal} | HNSW: {index_hnsw.ntotal} | Small: {index_small.ntotal}')

Flat: 40 | HNSW: 40 | Small: 40


### 5.1 FAISS explicado

**Búsqueda vectorial:** dado un vector query, encontrar los K vectores más cercanos por similitud coseno.

**Top-k:** los K resultados con mayor similitud. k bajo = preciso pero poco contexto. k alto = más recall, más ruido.

**Justificación k=5:** balance cobertura/precisión para preguntas factuales sobre noticias.

**HNSW (BONUS):** grafo multinivel. O(log N) vs O(N) Flat. M=32 vecinos, efSearch=64. Ventaja escala con corpus grandes.

## 6. Pipeline de Retrieval

In [8]:
def retrieve(query, index, corpus, k=5):
    qv = embed_query(query).astype('float32').reshape(1,-1)
    D, I = index.search(qv, k)
    return [(corpus[i], float(D[0][j])) for j, i in enumerate(I[0])]

q_test = '¿Qué dijo el gobierno sobre la economía?'
hits = retrieve(q_test, index_flat, corpus_large, k=5)
for c, s in hits:
    print(f"[{s:.3f}] {c['title'][:80]}")
    print(f"  {c['text'][:140]}...\n")

[0.790] Más de 100 mypes fueron capacitadas en herramientas tecnologías para incrementar
  Más de 100 mypes fueron capacitadas en herramientas tecnologías para incrementar ventas. El Ministerio de la Producción (Produce), a través ...

[0.786] INEI: Demanda interna impulsó el crecimiento del PBI en 3.5% en primer trimestre
  INEI: Demanda interna impulsó el crecimiento del PBI en 3.5% en primer trimestre de 2026. En el primer trimestre de 2026, el Producto Bruto ...

[0.782] Ministro de Trabajo suscribe memorando sobre migración laboral digna en Colombia
  Ministro de Trabajo suscribe memorando sobre migración laboral digna en Colombia. El ministro de Trabajo y Promoción del Empleo, Óscar Ferná...

[0.773] Sector pesquero amazónico: inversión de S/ 22 millones potenciará CiteProductivo
  Sector pesquero amazónico: inversión de S/ 22 millones potenciará CiteProductivo. El Ministerio de la Producción (Produce) anunció que se vi...

[0.772] INPE destaca proyectos de ley que reforman segur

## 7. Integración con LLM — Qwen2.5-3B-Instruct

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4'
)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map='auto', torch_dtype=torch.float16
)
print('Modelo cargado:', MODEL_ID)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo cargado: Qwen/Qwen2.5-3B-Instruct


In [10]:
SYSTEM = ("Eres un asistente periodístico. Responde en español ÚNICAMENTE con información "
          "de los pasajes dados. Si la respuesta no está en los pasajes, di: "
          "'No encontrado en el contexto.' Cita las fuentes como [chunk_id].")

def build_prompt(query, hits):
    ctx = '\n\n'.join([f"[{h[0]['chunk_id']}] {h[0]['text']}" for h in hits])
    msgs = [
        {'role':'system','content':SYSTEM},
        {'role':'user','content':f'Pasajes:\n{ctx}\n\nPregunta: {query}\nRespuesta:'}
    ]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def generate(query, hits, max_new=300):
    prompt = build_prompt(query, hits)
    inp = tok(prompt, return_tensors='pt').to(llm.device)
    out = llm.generate(**inp, max_new_tokens=max_new, do_sample=False, repetition_penalty=1.05)
    return tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

def rag(query, k=5, index=index_flat, corpus=corpus_large):
    hits = retrieve(query, index, corpus, k=k)
    ans = generate(query, hits)
    return ans, hits

ans, hits = rag('¿Qué dijo el gobierno sobre la economía peruana?')
print(ans)

No encontrado en el contexto.


## 8. Grounding básico
Verifica qué proporción de frases generadas tiene soporte léxico en los chunks recuperados.

In [11]:
from nltk.tokenize import sent_tokenize
import re

def normalize(s):
    return re.sub(r'[^a-záéíóúñ0-9 ]', ' ', s.lower())

def ngrams(tokens, n=4):
    return set(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def grounding_ratio(answer, hits, n=4):
    ctx = ' '.join([h[0]['text'] for h in hits])
    ctx_ng = ngrams(normalize(ctx).split(), n)
    sents = sent_tokenize(answer, language='spanish')
    if not sents: return 0.0, []
    flags = []
    for s in sents:
        s_ng = ngrams(normalize(s).split(), n)
        overlap = len(s_ng & ctx_ng) / max(len(s_ng), 1)
        flags.append((s, overlap, overlap > 0.15))
    supported = sum(1 for _,_,ok in flags if ok)
    return supported/len(sents), flags

ratio, detail = grounding_ratio(ans, hits)
print(f'Grounding ratio: {ratio:.2%}')
for s, o, ok in detail:
    mark = 'OK' if ok else 'NO'
    print(f'[{mark}] ({o:.2f}) {s[:120]}')

Grounding ratio: 0.00%
[NO] (0.00) No encontrado en el contexto.


## 9. Evaluación del sistema
Métricas: Recall@k, Precision@k, Grounding ratio.

In [12]:
eval_set = [
    {'q':'¿Qué medidas anunció el gobierno peruano sobre economía?', 'terms':['gobierno','economía','medida']},
    {'q':'¿Qué pasó con el Congreso esta semana?', 'terms':['congreso','sesión','ley']},
    {'q':'¿Cuál fue la última declaración del presidente?', 'terms':['presidente','declaró','dijo']},
    {'q':'¿Qué noticias hay sobre inflación en Perú?', 'terms':['inflación','precio','BCR']},
    {'q':'¿Qué pasó en las regiones del Perú?', 'terms':['región','regional','provincia']},
    {'q':'¿Hubo protestas recientes?', 'terms':['protesta','manifestación','movilización']},
    {'q':'¿Qué dijo el ministro de Economía?', 'terms':['ministro','economía','MEF']},
    {'q':'¿Qué noticias internacionales destacan?', 'terms':['internacional','exterior','país']},
]

def hit_relevant(text, terms):
    t = text.lower()
    return sum(1 for x in terms if x.lower() in t) >= max(1, len(terms)//2)

results = []
for ex in eval_set:
    hits = retrieve(ex['q'], index_flat, corpus_large, k=5)
    rel = [hit_relevant(h[0]['text'], ex['terms']) for h in hits]
    recall = 1.0 if any(rel) else 0.0
    precision = sum(rel)/len(rel)
    ans = generate(ex['q'], hits, max_new=200)
    g, _ = grounding_ratio(ans, hits)
    results.append({'q':ex['q'][:45], 'recall@5':recall, 'precision@5':precision, 'grounding':g})

df_eval = pd.DataFrame(results)
print(df_eval.to_string(index=False))
print(f'\nPromedio Recall@5: {df_eval["recall@5"].mean():.2%}')
print(f'Promedio Precision@5: {df_eval["precision@5"].mean():.2%}')
print(f'Promedio Grounding: {df_eval["grounding"].mean():.2%}')

                                            q  recall@5  precision@5  grounding
¿Qué medidas anunció el gobierno peruano sobr       0.0          0.0        0.0
       ¿Qué pasó con el Congreso esta semana?       1.0          0.2        0.5
¿Cuál fue la última declaración del president       1.0          0.2        0.0
   ¿Qué noticias hay sobre inflación en Perú?       0.0          0.0        0.0
          ¿Qué pasó en las regiones del Perú?       1.0          0.2        0.6
                   ¿Hubo protestas recientes?       0.0          0.0        0.0
           ¿Qué dijo el ministro de Economía?       1.0          0.2        0.0
      ¿Qué noticias internacionales destacan?       1.0          0.6        0.0

Promedio Recall@5: 62.50%
Promedio Precision@5: 17.50%
Promedio Grounding: 13.75%


## 10. Análisis de resultados

**Casos buenos esperados:** preguntas con entidades nombradas peruanas (BCR, MEF, Congreso) → retrieval preciso por coincidencia léxica + semántica.

**Casos malos esperados:** preguntas muy genéricas ("¿qué noticias hay?") → recupera chunks dispersos sin foco.

**Decisiones técnicas:**
- Chunk 256/48: noticias tienen pirámide invertida, contexto inicial denso.
- k=5: balance entre cobertura y ruido en prompt.
- Multilingual-e5-base: nativo en español, no requiere traducción.
- Qwen2.5-3B 4-bit: español competente, cabe en T4 16GB.

**Limitaciones:**
- Dataset perecedero (noticias actuales).
- Anti-bot puede limitar scraping a 100-120 artículos.
- Grounding lexical, no semántico.
- Evaluación con relevancia heurística por keywords.

**Mejoras futuras:**
- Reranker cross-encoder en español.
- Hybrid BM25+FAISS (BM25 brilla con nombres propios peruanos).
- BLEU/ROUGE contra el cuerpo original de la noticia.
- Más feeds (RPP, La República) para diversificar fuente.